# Async TMDB Lookup — Prototype

Prototype for the async rewrite of `scripts/fetch_tmdb_metadata.py`.

**Roadmap:**
1. **Part 1** — Stress-test the `RateLimiter` class in isolation to verify it holds to TMDB's 40 req / 10 s limit
2. **Part 2** — Build `search_movie` with year-fallback
3. **Part 3** — End-to-end: load ratings, run all lookups concurrently, join against MovieLens

**References:**
- [TMDB rate limiting docs](https://developer.themoviedb.org/docs/rate-limiting) — 40 req / 10 s
- [`httpx` async client](https://www.python-httpx.org/async/) — async-native drop-in for `requests`
- [`asyncio.Semaphore`](https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore) — caps concurrency and backs the rate limiter

In [13]:
import asyncio
import os
import time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path('../.env'))
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_BASE    = 'https://api.themoviedb.org/3'

MAX_CONCURRENT = 10    # simultaneous in-flight requests
RATE_LIMIT     = 40    # requests per window
RATE_PERIOD    = 10.0  # window size in seconds

assert TMDB_API_KEY, 'Set TMDB_API_KEY in .env'
print('API key loaded ✓')

API key loaded ✓


## Part 1 — Rate Limiter

A [sliding-window rate limiter](https://www.figma.com/blog/an-alternative-approach-to-rate-limiting/) built on `asyncio.Semaphore`.

**Idea:** initialise the semaphore with `rate` tokens. Each acquired token is returned to the pool exactly `period` seconds later via [`loop.call_later`](https://docs.python.org/3/library/asyncio-eventloop.html#asyncio.loop.call_later) (a synchronous callback — safe to call on `Semaphore.release`). At any moment, *tokens in use = requests made in the last `period` seconds ≤ rate*.

In [14]:
class RateLimiter:
    """
    Sliding-window rate limiter backed by asyncio.Semaphore.

    Acquiring a token marks the start of a request; the token is
    automatically returned `period` seconds later, ensuring at most
    `rate` requests in any rolling `period`-second window.

    Docs: https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore
    """

    def __init__(self, rate, period):
        self._sem    = asyncio.Semaphore(rate)
        self._period = period

    async def __aenter__(self):
        await self._sem.acquire()
        return self

    async def __aexit__(self, *_):
        # Schedule token release after `period` seconds.
        # call_later uses a synchronous callback — Semaphore.release() qualifies.
        loop = asyncio.get_running_loop()
        loop.call_later(self._period, self._sem.release)

### Stress test

Fire **25 tasks** through `RateLimiter(rate=10, period=3.0)` and print when each one acquires its token.

Expected pattern:
| Batch | Tasks | Acquires at |
|-------|-------|-------------|
| 1st   | 0–9   | ~0 s        |
| 2nd   | 10–19 | ~3 s        |
| 3rd   | 20–24 | ~6 s        |

In [15]:
async def stress_test(rate=10, period=3.0, n_tasks=25):
    limiter = RateLimiter(rate=rate, period=period)
    records = []
    t0 = time.monotonic()

    async def acquire_and_log(i):
        async with limiter:
            elapsed = time.monotonic() - t0
            records.append((i, elapsed))
            await asyncio.sleep(0)  # yield; simulate instant 'request'

    await asyncio.gather(*[acquire_and_log(i) for i in range(n_tasks)])

    records.sort(key=lambda x: x[1])
    print(f"{'task':>5}  {'acquired':>10}  batch")
    print("-" * 40)
    for task_i, t in records:
        batch  = int(t / period)
        bar    = "█" * (batch + 1)
        print(f"{task_i:5d}  {t:8.2f}s  {bar}")

    total        = time.monotonic() - t0
    batches      = (n_tasks - 1) // rate
    expected_min = batches * period
    ok = "✓" if total >= expected_min - 0.1 else "✗"
    print(f"\nTotal: {total:.1f}s  |  Expected ≥ {expected_min:.0f}s  {ok}")


await stress_test()

 task    acquired  batch
----------------------------------------
    0      0.00s  █
    1      0.00s  █
    2      0.00s  █
    3      0.00s  █
    4      0.00s  █
    5      0.00s  █
    6      0.00s  █
    7      0.00s  █
    8      0.00s  █
    9      0.00s  █
   10      3.02s  ██
   11      3.02s  ██
   12      3.02s  ██
   13      3.02s  ██
   14      3.02s  ██
   15      3.02s  ██
   16      3.02s  ██
   17      3.02s  ██
   18      3.02s  ██
   19      3.02s  ██
   20      6.03s  ███
   21      6.03s  ███
   22      6.03s  ███
   23      6.03s  ███
   24      6.03s  ███

Total: 6.0s  |  Expected ≥ 6s  ✓


## Part 2 — `search_movie` Coroutine

Each movie may require 1–2 TMDB API calls:
1. Search with `primary_release_year` (year-scoped, more precise)
2. If empty → retry with bare title only (year fallback)

Both the concurrency `Semaphore` and the `RateLimiter` wrap each individual `client.get(...)` call, so every HTTP request — including fallback retries — respects both limits independently.

Reference: [TMDB search/movie endpoint](https://developer.themoviedb.org/reference/search-movie)

In [16]:
async def search_movie(client, concurrency, rate, title, year, rating):
    """
    Search TMDB for a single movie.
    Tries primary_release_year first; falls back to bare title if results are empty.
    Returns a metadata dict on success, None if nothing is found.
    """
    base_params = {
        'api_key':       TMDB_API_KEY,
        'query':         title,
        'include_adult': False,
    }

    async def fetch(params):
        async with concurrency:
            async with rate:
                try:
                    resp = await client.get(f'{TMDB_BASE}/search/movie', params=params)
                    resp.raise_for_status()
                    return resp.json().get('results', [])
                except httpx.HTTPError as exc:
                    print(f'  [warn] {title!r}: {exc}')
                    return []

    results = await fetch({**base_params, 'primary_release_year': year})
    if not results:
        results = await fetch(base_params)   # fallback: no year filter
    if not results:
        return None

    best = results[0]
    return {
        'letterboxd_title': title,
        'tmdb_id':          best['id'],
        'original_title':   best.get('original_title'),
        'release_year':     (best.get('release_date') or '')[:4] or None,
        'rating':           rating,
    }

## Part 3 — End-to-End

1. Pre-load `links.csv` as a `tmdbId → movieId` dict (avoids a pandas merge later)
2. Read `ratings.csv` → list of `(title, year, rating)` tuples
3. Fire all lookups concurrently with [`asyncio.gather`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather)
4. Partition results into matched / ML-unmatched / not-on-TMDB

In [17]:
data_path = Path('../data')

# Pre-load tmdbId → movieId lookup
# MovieLens 32M links.csv: https://grouplens.org/datasets/movielens/32m/
links      = pd.read_csv(data_path / 'ml-32m' / 'links.csv')
tmdb_to_ml = dict(zip(links['tmdbId'], links['movieId']))
print(f'Loaded {len(tmdb_to_ml):,} tmdbId → movieId mappings')

# Load Letterboxd ratings
ratings = pd.read_csv(data_path / 'ratings.csv')
rows = [
    (str(r.Name), int(r.Year) if pd.notna(r.Year) else None, float(r.Rating))
    for r in ratings.itertuples(index=False)
]
print(f'Ratings to look up: {len(rows)}')
rows[:3]

Loaded 87,549 tmdbId → movieId mappings
Ratings to look up: 69


[('Wicked: For Good', 2025, 2.0),
 ('Novocaine', 2025, 2.5),
 ('Zootopia', 2016, 4.0)]

In [18]:
concurrency = asyncio.Semaphore(MAX_CONCURRENT)
rate        = RateLimiter(RATE_LIMIT, RATE_PERIOD)

t0 = time.monotonic()

async with httpx.AsyncClient(timeout=15.0) as client:
    tasks   = [
        search_movie(client, concurrency, rate, title, year, rating)
        for title, year, rating in rows
    ]
    results = await asyncio.gather(*tasks)

elapsed = time.monotonic() - t0
print(f'Completed {len(rows)} lookups in {elapsed:.1f}s  '
      f'({len(rows)/elapsed:.1f} effective req/s)')

Completed 69 lookups in 10.5s  (6.6 effective req/s)


In [19]:
tmdb_records = []
matched      = []
ml_unmatched = []
not_on_tmdb  = []

for (title, _, _), result in zip(rows, results):
    if result is None:
        not_on_tmdb.append(title)
        continue
    tmdb_records.append(result)
    movie_id = tmdb_to_ml.get(result['tmdb_id'])
    if movie_id is not None:
        matched.append({'movieId': movie_id, **result})
    else:
        ml_unmatched.append(result)

print(f"{'Your ratings:':<25} {len(rows)}")
print(f"{'Found on TMDB:':<25} {len(tmdb_records)}")
print(f"{'Matched in ml-32m:':<25} {len(matched)}")
print(f"{'No MovieLens entry:':<25} {len(ml_unmatched)}")
print(f"{'Not found on TMDB:':<25} {len(not_on_tmdb)}")

Your ratings:             69
Found on TMDB:            68
Matched in ml-32m:        32
No MovieLens entry:       36
Not found on TMDB:        1


In [20]:
matched_df = pd.DataFrame(matched)
matched_df[['letterboxd_title', 'movieId', 'rating', 'release_year']].head(10)

,letterboxd_title,movieId,rating,release_year
0,Zootopia,152081,4.0,2016
1,Road Trip,3617,3.5,2000
2,The Terminal,8529,3.0,2004
3,Sonic the Hedgehog,210577,2.5,2020
4,Sonic the Hedgehog 2,271799,3.0,2022
5,Titanic,1721,3.5,1997
6,Chef,111443,3.5,2014
7,Coherence,113741,4.0,2014
8,Avatar,72998,3.5,2009
9,A Rainy Day in New York,198157,2.5,2019


In [21]:
if not_on_tmdb:
    print(f'Not found on TMDB ({len(not_on_tmdb)}):')
    for t in not_on_tmdb:
        print(f'  - {t}')
else:
    print('All ratings found on TMDB ✓')

if ml_unmatched:
    print(f'\nFound on TMDB but missing from MovieLens ({len(ml_unmatched)}) — likely recent releases:')
    for r in ml_unmatched:
        print(f"  - {r['letterboxd_title']} ({r['release_year']})")

Not found on TMDB (1):
  - BLUE EYE SAMURAI

Found on TMDB but missing from MovieLens (36) — likely recent releases:
  - Wicked: For Good (2025)
  - Novocaine (2025)
  - Zootopia 2 (2025)
  - Wake Up Dead Man (2025)
  - Sonic the Hedgehog 3 (2024)
  - Bugonia (2025)
  - Avatar: Fire and Ash (2025)
  - 100 METERS (2025)
  - Predator: Badlands (2025)
  - People We Meet on Vacation (2026)
  - Eternity (2025)
  - Marty Supreme (2025)
  - The Wrecking Crew (2026)
  - GOAT (2026)
  - Hamnet (2025)
  - Death of a Unicorn (2025)
  - KPop Demon Hunters (2025)
  - F1 (2025)
  - Roofman (2025)
  - Train Dreams (2025)
  - No Other Choice (2025)
  - One Battle After Another (2025)
  - Sentimental Value (2025)
  - The Secret Agent (2025)
  - Nosferatu (2024)
  - HAIKYU!! The Dumpster Battle (2024)
  - Sinners (2025)
  - Hoppers (2026)
  - The Super Mario Galaxy Movie (2026)
  - Spider-Man: Homecoming (2017)
  - Project Hail Mary (2026)
  - How to Make a Killing (2026)
  - Anyone But You (2023)
  - M

In [22]:
# Save outputs — same files the final script will produce
pd.DataFrame(tmdb_records).to_csv(data_path / 'tmdb_metadata.csv',      index=False)
pd.DataFrame(matched)     .to_csv(data_path / 'movielens_matched.csv',  index=False)
pd.DataFrame(ml_unmatched).to_csv(data_path / 'movielens_unmatched.csv', index=False)

print('Saved:')
print(f'  tmdb_metadata.csv       ({len(tmdb_records)} rows)')
print(f'  movielens_matched.csv   ({len(matched)} rows)')
print(f'  movielens_unmatched.csv ({len(ml_unmatched)} rows)')

Saved:
  tmdb_metadata.csv       (68 rows)
  movielens_matched.csv   (32 rows)
  movielens_unmatched.csv (36 rows)
